# 텐서 이해하기

In [111]:
import torch

tensor0d = torch.tensor(1)

tensor1d = torch.tensor([1,2,3])

tensor2d = torch.tensor([[1,2,3],[4,5,6]])

tensor3d = torch.tensor([[[1,2],[3,4]],
                         [[5,6],[7,8]]])

# 정수 데이터 타입 - 64비트
print(tensor1d.dtype)

# 파이썬 부동 소수점 - 32비트 정밀도 텐서
floatvec = torch.tensor([1.0,2.0,3.0])
print(floatvec.dtype)

torch.int64
torch.float32


# 자주 사용하는 파이토치 텐서 연산

In [112]:
print(tensor2d)

# 텐서의 크기 확인
print(tensor2d.shape)

# 텐서의 크기 바꾸기
print(tensor2d.reshape(3,2))

# view를 통해서 텐서의 크기를 바꾸기, 데이터가 연속적으로 놓여있어야 함!
print(tensor2d.view(2,3))

# 행렬의 곱
print(tensor2d.matmul(tensor2d.T))
# = print(tensor2d @ tensor2d.T)과 동일

tensor([[1, 2, 3],
        [4, 5, 6]])
torch.Size([2, 3])
tensor([[1, 2],
        [3, 4],
        [5, 6]])
tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[14, 32],
        [32, 77]])


# 모델을 계산 그래프로 보기
## 로지스틱 회귀의 정방향 계산

파이토치로 계산을 수행하는 과정에서, 연산에 사용되는 리프 노드 중 하나라도 requires_grad 속성이 True라면,
기본적으로 그 연산의 결과에 대한 계산 그래프가 생성된다.
이 그래프는 역전파(backpropagation)를 수행하기 위해 필요한 것으로,
requires_grad=True는'역전파를 수행할 수 있도록 준비(기록)한다'는 의미로 이해하면 된다!


In [113]:
import torch.nn.functional as F
from torch.autograd import grad
import time

y = torch.tensor([1.0]) # 정답 레이블
x1 = torch.tensor([1.1]) # 입력 특성
w1 = torch.tensor([2.2], requires_grad=True) # 가중치 파라미터
b = torch.tensor([0.0], requires_grad=True) # 편향 유닛
z = x1 * w1 + b # 순 입력
a = torch.sigmoid(z) # 활성화 함수와 출력

# 자동 미분을 손쉽게

## autograd로 그레디언트 계산하기

In [114]:
loss = F.binary_cross_entropy(a,y)

# grad 함수를 통해서 모델 파아미터에 대한 손실 그레디언트를 계산
grad_L_w1 = grad( loss, w1, retain_graph=True)
grad_L_b = grad( loss, b, retain_graph=True)

print(grad_L_w1, grad_L_b)

# .backward 메서드를 호출
# 파이로치가 그래프에 있는 모든 리프 노드의 그레디언트를 계산해서 텐서의 .grad에 저장하기
loss.backward()
print(w1.grad, b.grad)

(tensor([-0.0898]),) (tensor([-0.0817]),)
tensor([-0.0898]) tensor([-0.0817])


# 다층 신경망 만들기

ReLU : https://en.wikipedia.org/wiki/Rectified_linear_unit

In [115]:
# 2개의 은닉층을 가진 다층 퍼셉트론
class NeuralNetwork(torch.nn.Module):
    def __init__(self,num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 1번째 은닉층 / Linear층은 입력개수와 출력 개수를 매개변수로 가진다.
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2번째 은닉층 / 출력 노드 개수는 다음 층의 입력 노드 개수와 동일 해야함
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # 출력층
            torch.nn.Linear(20,num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits


# 모델의 구조 출력
model = NeuralNetwork(50, 3)
print(model)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("\n훈련 가능한 모델의 총 파라미터 개수: ",num_params)

print("\n",model.layers[0].weight) # 신경망 첫 번째 층의 가중치 행렬 전체의 값과 형태를 출력
print(model.layers[0].weight.shape)
print("편향 백터: ", model.layers[0].bias) # 편향 벡터 값을 출력

# 대칭성을 깨트리기 위해 모델 가중치를 작은 난수로 초기화하기
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print("\n",model.layers[0].weight)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

훈련 가능한 모델의 총 파라미터 개수:  2213

 Parameter containing:
tensor([[-0.0281,  0.1284,  0.1009,  ..., -0.0914, -0.1354,  0.1046],
        [ 0.1062, -0.0360,  0.0624,  ...,  0.0697, -0.0964,  0.0765],
        [-0.0045,  0.0584, -0.0003,  ..., -0.1287, -0.0371, -0.1072],
        ...,
        [-0.1057,  0.1387, -0.1316,  ...,  0.0019, -0.0163,  0.0589],
        [-0.1053, -0.0406, -0.1302,  ...,  0.0139, -0.0647, -0.0738],
        [-0.0511,  0.0880,  0.0709,  ...,  0.1037,  0.0555,  0.0037]],
       requires_grad=True)
torch.Size([30, 50])
편향 백터:  Parameter containing:
tensor([ 0.0560, -0.0131,  0.1141,  0.0025,  0.0788,  0.1260,  0.0459,  0.0039,
        -0.0099, -0.1152, -0.1105,  0.0472,  0.0131,  0.0366, -0.1277,  0.0792,
        -0.0146,  0

In [116]:
# 정방량 계산에서 신경망을 사용하는 방법
torch.manual_seed(123)
X = torch.rand((1, 50)) # 랜덤란 훈련 샘플 X, 50개 차원 특성 백터를 기대
out = model(X)
print(out)
# 결과 ; tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)
# AddmmBackward0 -> addmm 행렬곱셈 이후 덧셈이 있었다

# 모델의 추론이 아닌 추론의 경우에는 -> no_grad() 사용하기 => 파이토치에게 그레디언트 추적 필요 없음을 알림!
with torch.no_grad():
    out = model(X)
print(out)
'''
모델의 훈련 : 가중치와 편향 지속적으로 수정
이때 그레디언트(Gradient, 기울기)는 오차를 줄이려면 파라미터를 어느 방향으로, 얼마나 수정해야 하는지 알려줌

추론 : 학습이 끝난 모델에 새로운 데이터를 넣고 결과를 예측하는 과정
이때는 이미 최적화된 가중치를 고정해 두고 사용, 오차를 계산해 파라미터 업데이트 할 이유 X
'''

with torch.no_grad():
    out = torch.softmax(model(X), dim = 1)
print(out)
# 모두 다 더하면 1이됨 -> 클래스(정답 후보) 소속 확률

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)
tensor([[-0.1262,  0.1080, -0.1792]])
tensor([[0.3113, 0.3934, 0.2952]])


# 효율적인 데이터 로더 설정하기

In [117]:
# 예시 데이터셋 만들기
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

# 2개의 샘플로 구성된 테스트 세트 만들기
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])

# Dataset 클래스를 이용해서, 사용자 정의 데이터셋 클래스 ToyDataset을 만들기
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    # 하나의 데이터 레코드와 이에 해당하는 레이블을 추출
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    # 데이터 셋의 총 길이를 반환
    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

print(len(train_ds), "\n\n")
# 아까 훈련데이터 셋 5개의 행을 가지고 있었음!

# 데이터 로더 초기화
from torch.utils.data import DataLoader

torch.manual_seed(123)

# 트레인 셋 데이터로더
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True # 훈련 데이터 로더를 순회하면서 마지막 배치를 제외
)

# 테스트 셋 데이터로더
test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

# 데이터 로더 순회하기
for idx, (x,y) in enumerate(train_loader):
    print(f"배치 {idx+1}:", x, y)
# 정확히 한 번씩 훈련 샘플을 방문 -> 이것을 에포크라고 부름!

# num_workers를 0보다 크게 설정하면 여러 개의 워커 프로세스가 병렬로 데이터를 로드하므로 메인 프로세스가 모델 훈련에만 집중하게 되고 시스템의 자원을 잘 활용할 수 있음

5 


배치 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
배치 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])


# 일반적인 훈련 루프

In [118]:
# 파이토치에서 신경망 훈련하기
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2) # 2개의 특성과 2개의 클래스(정답의 종류가 2가지임을 의미한다!)를 가진다
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.5
) # 경사하강법을 통해서 옵티마이저(옵티마이저는 모델이 문제를 풀고 틀렸을 때, 가중치를 어느 방향으로 얼마나 수정해야 할지를 결정 + 실제로 업데이트)에게 최적화할 파라미터를 전달하기

# 훈련을 3회
num_epochs = 3
for epoch in range(num_epochs):

    print("\n")
    model.train()
    # 훈련 테이터 로더 순회하면서 특징과 라벨 뽑아오기
    for batch_idx, (features, labels) in enumerate(train_loader):
        # 뽑은 특징을 모델에 넣어서 모델이 예측한 값을 뽑아내는 과정!
        logits = model(features)

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward() # 손실 그레디언트 계산
        optimizer.step() # 옵티마이저가 파라미터 업데이트

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()



Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65


Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13


Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


### 하이퍼 파라미터
- 학습률 (Learning Rate, LR) : SGD에서 얼마나 이동할까요!
- 배치 사이즈 (batch size) : 한 번의 스택에서 동시에 처리할 데이터의 묶음 개수
- epochs 수 : 전체 훈련 데이터셋을 몇 번 반복해서 학습할 것인지
- 옵티마이저 type : 가중치를 어떤 방식으로 업데이트 할 것인지
    - SGD, Adam, 등

### 검증 데이터 셋과 테스트 데이터 셋
- 검증 데이터셋은 모델이 학습을 진행하는 도중에 중간 점검을 하기 위한 데이터로서, 모델 설정을 조정(최적의 하이퍼파라미터 설정)하기 위해 여러 번 사용
- 테스트 데이터셋은 모델의 모든 학습이 완전히 끝난 후, 진짜 실력을 딱 한번 평가하기 위한 데이터셋을 말한다.

In [119]:
with torch.no_grad():
    outputs = model(X_train)
print(outputs) # 모델을 훈련시킨 이후 이를 사용해서 예측을 만들기

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [120]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas) # 클래스 소속 확률 구하기 위해서 소프트 맥스 함수 사용하기
''' 출략에 있는 첫번째 항을 보면, 이 훈련 샘플이 0에 속할 확률을 의미하고,
클래스 1에 속할 확률이 0.09%라는 의미임! -> 이것을 좀 더 편하게 보자!'''

# 소프트 맥스로 구한 것에 argmax 씌우기
predictions = torch.argmax(probas, dim=1)
print(predictions)

# 그냥 로짓에 argmax를 씌우기
predictions = torch.argmax(outputs, dim=1)
print(predictions)

# 올바은 예측의 개수를 확인하기
torch.sum(predictions == y_train)

tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])
tensor([0, 0, 0, 1, 1])
tensor([0, 0, 0, 1, 1])


tensor(5)

In [121]:
# 예측 정확도 계산을 위한 일반화 함수

def compute_accuracy(model, dataloader):
    model = model.eval()
    correct, total_examples = 0.0, 0

    # 데이터 로더 순회하면서
    for idx, (features, labels) in enumerate(dataloader):

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions # 레이블 일치 불일치 값을 저장
        correct += torch.sum(compare) # 레이블 일치의 개수를 구하기
        total_examples += len(compare) # 전체 예측 개수 구하기

    return (correct/total_examples).item()

# 훈련 데이터 셋에 적용 시키기
print(compute_accuracy(model, train_loader))

# 테스트 데이터 셋에 적용 시키기
print(compute_accuracy(model, test_loader))

1.0
1.0


# 모델 저장과 로드

In [122]:
# 파이토치에서 모델을 저장하고 로드할 때 추천하는 방법
torch.save(model.state_dict(),"model.pth") # model.pth는 디스크에 저장하기 위한 파일의 이름을 가리킴

# 저장한 뒤에는 가져오기 가능
model = NeuralNetwork(num_inputs=2, num_outputs=2)
model.load_state_dict(torch.load("model.pth"))

<All keys matched successfully>

GPU로 훈련 성능 최적화하기

In [123]:
# GPU 연산 지원을 확인
print(torch.cuda.is_available())

# 텐서 2개 더하기
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])
print(tensor_1 + tensor_2)

print("\n텐서를 GPU로 이동한 후 덧셈 연산을 수행하기")
tensor_1 = tensor_1.to("cuda")
tensor_2 = tensor_2.to("cuda")

print(tensor_1 + tensor_2) # device='cuda:0' -> 텐서가 첫 번째 GPU에 있다는 것을 의미

True
tensor([5., 7., 9.])

텐서를 GPU로 이동한 후 덧셈 연산을 수행하기
tensor([5., 7., 9.], device='cuda:0')


In [124]:
### 단일 GPU를 사용하는 훈련 루프
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # GPU로 장치 변수를 정의하기
model = model.to(device) # 모델을 GPU로 전송하기

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        features, labels = features.to(device), labels.to(device) # 데이터를 GPU로 전송하기
        logits = model(features)
        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00
